In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import ttest_ind, mannwhitneyu, zscore
from statsmodels.stats.multitest import multipletests

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
clusters = pd.read_csv(f'{projdir}/csv/clusters/rna_leiden_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata.obs['coarse'] = clusters['coarse_v7'].reindex(adata.obs.index)
adata.obs['coarse_donor'] = [f'{x}_{y}' for x,y in zip(adata.obs['coarse'], adata.obs['assignment'])]

In [ ]:
def process_pseudo(s):
    tmp_adata = adata[adata.obs['coarse_donor'] == s]
    mtx = tmp_adata.X.sum(axis=0)
    return mtx

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_pseudo, clusts), total=len(clusts)))
    
pseudo_merged = np.vstack(results)
pseudo_df = pd.DataFrame(pseudo_merged, index=clusts, columns=adata.var.index)
pseudo_df.shape

In [ ]:
pseudo_df.head()

In [ ]:
denom = pseudo_df.sum(axis=1)
norm_pseudo = pseudo_df.div(denom, axis=0) * 1e6
norm_pseudo.shape

In [ ]:
norm_pseudo.head()

In [ ]:
z_pseudo = norm_pseudo.apply(zscore).fillna(0)

In [ ]:
z_pseudo.head()

In [ ]:
# # tmp_clusts = [f'Start_{x}' for x in donors]

In [ ]:
pval_cutoff = 0.01
fc_cutoff = 1
filt_results = []
    
for clust in clusts:
    tmp_degs = pd.read_csv(f'{projdir}/csv/degs/by_donor/{clust}.csv', sep='\t', index_col=0)
    mask = (tmp_degs['pvals_adj'] < pval_cutoff) & (tmp_degs['logfoldchanges'] > fc_cutoff)
    final_degs = tmp_degs[mask]
    filt_results.append(final_degs)

In [ ]:
donor_deg_results = []

for i in range(4):
    tmp_degs = filt_results[i]
    tmp_degs = tmp_degs[tmp_degs['names'].isin(z_pseudo.columns)]
    top_degs = tmp_degs['names']
    donor_deg_results.append(z_pseudo.loc[:,top_degs])

In [ ]:
i = 0
tmp_df = pd.DataFrame(donor_deg_results[i]).T
to_plot = tmp_df.melt(value_vars=clusts)
to_plot['clust'] = [x.split('_')[0] for x in to_plot['variable']]
to_plot['donor'] = [x.split('_')[1] for x in to_plot['variable']]
to_plot.shape

In [ ]:
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'clust'
y_lab = 'value'
hue = 'donor'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
tmp_results = []
for i in range(4):
    s = donors[i]
    tmp_df = pd.DataFrame(donor_deg_results[i]).T
    to_plot = tmp_df.melt(value_vars=clusts)
    to_plot['clust'] = [x.split('_')[0] for x in to_plot['variable']]
    to_plot['donor'] = [x.split('_')[1] for x in to_plot['variable']]

    ###
    fig, axes = plt.subplots(1, figsize=(8, 4))

    x_lab = 'clust'
    y_lab = 'value'
    hue = 'donor'

    ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
    ax.set_xlabel('', fontsize=12)
    ax.set_ylabel('', fontsize=12)
    ax.set_title(f'DEGs from Start_{s}', fontsize=16)

    ax.tick_params(axis='x', labelsize=14, labelrotation=0)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    to_plot['plot_donor'] = 'rest'
    mask = to_plot['donor'] == s
    to_plot.loc[mask, 'plot_donor'] = 'target'
    tmp_results.append(to_plot)

In [ ]:
    
#     ###

In [ ]:
target_plot = pd.concat(tmp_results)

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 4))

x_lab = 'clust'
y_lab = 'value'
hue = 'plot_donor'

sns.boxplot(data=target_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False, ax=ax)
ax.set_xlabel('', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title(f'DEGs over trajectory', fontsize=16)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

### put a p-value to it?

In [ ]:
all_pvals = []
for i in range(4):
    
    donor = donors[i]
    tmp_df = donor_deg_results[i]

    tmp_pvals = []

    for s in tmp_order:
        mask = [x.split('_')[0] == s for x in tmp_df.index]
        clust_df = tmp_df.loc[mask,:]
        mask = np.array([x.split('_')[1] == donor for x in clust_df.index])
        donor_vals = clust_df.loc[mask].values.reshape(-1,1)
        rest_vals = clust_df.loc[~mask].mean(axis=0).values.reshape(-1,1)
        tmp_pvals.append(ttest_ind(a=donor_vals, b=rest_vals).pvalue[0])
        
    all_pvals.append(tmp_pvals)
pd.DataFrame(all_pvals, index=donors, columns=tmp_order)

In [ ]:
all_pvals = []
for i in range(4):
    
    donor = donors[i]
    tmp_df = donor_deg_results[i]

    tmp_pvals = []

    for s in tmp_order:
        mask = [x.split('_')[0] == s for x in tmp_df.index]
        clust_df = tmp_df.loc[mask,:]
        mask = np.array([x.split('_')[1] == donor for x in clust_df.index])
        donor_vals = clust_df.loc[mask].values.reshape(-1,1)
        rest_vals = clust_df.loc[~mask].mean(axis=0).values.reshape(-1,1)
        tmp_pvals.append(mannwhitneyu(donor_vals, rest_vals).pvalue[0])
        
    all_pvals.append(tmp_pvals)
    
pval_df = pd.DataFrame(all_pvals, index=donors, columns=tmp_order)
pval_df.head()

### what about merging across donors

In [ ]:
# ### everything suddenly becomes significant... I'm not sure if this is correct

#     #         donor_vals = clust_df.loc[mask].values.reshape(-1,1)

    
# # pd.DataFrame(all_pvals, index=donors, columns=tmp_order)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'

target_plot.to_csv(f'{outdir}/rna_donor_deg_traj.csv', sep='\t')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'

pval_df.to_csv(f'{outdir}/rna_donor_deg_traj.pvals.csv', sep='\t')

In [ ]:
!date